In [1]:
"""
plot_day_hawkes.py -- day visualisation of the Stage 2 Hawkes results.

PRECONDITION
  raw must carry a bar_index column identical to Stage 0's (positional in the
  >=09:30 filtered, timestamp-sorted file). All joins are on bar_index; no
  timestamp arithmetic happens here.

PANES
  1  OHLC + JMA coloured by leg direction, JMA dots coloured by calibrated
     next-bar hazard, markers at true JMA extrema.
  2  jmaD1 / jmaD2 with pivot markers (red = opposing the leg, green = confirming).
  3  tickJmaD1 / tickJmaD2, same marker convention.
  4  Hazard pane: calibrated per-bar hazard, frozen-path pbreak(H), direct
     within-H break probability. Dotted line = unconditional base rate.

NOTES
  P1  A target event at bar t means the JMA extremum sits at bar t-1. Markers are
      drawn at the extremum (what the eye sees); the turn is only knowable at t.
  P2  pcont_hawkes uses the raw (uncalibrated) model internally, so pbreak_frozen
      is on the raw scale. Meaningful for H = 1..3 only.
  P3  ~395 target pivots per 6s session. Always slice a window; a whole day is
      unreadable.
"""

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

BASE_RATE = 0.10302

In [2]:
def build_day_frame(day, raw, bars, events, pred=None, pred8=None, res_frozen=None):
    #b = bars[bars["date"] == day]
    d = raw.merge(b[["bar_index", "jma_leg_dir", "leg_age", "is_target", "warm"]], on="bar_index", how="inner")

    if pred is not None:
        col = "p_cal" if "p_cal" in pred.columns else "p"
        d = d.merge(pred[["bar_index", col]].rename(columns={col: "hazard"}), on="bar_index", how="left")
    if pred8 is not None:
        d = d.merge(pred8[["bar_index", "p8"]], on="bar_index", how="left")
    if res_frozen is not None:
        d = d.merge(res_frozen[["bar_index", "pbreak"]].rename(columns={"pbreak": "pbreak_frozen"}), on="bar_index", how="left")

    ev = events[events["session_date"].astype(str) == day]
    return d.sort_values("bar_index").reset_index(drop=True), ev

def _seg(y, mask):
    out = np.full(len(y), np.nan)
    out[mask] = y[mask]
    return out

def _markers(fig, ev, stream, ts, row, size=7):
    e = ev[ev["stream"] == stream]
    if len(e) == 0:
        return
    for opp, colour, nm in ((1, "#d62728", "opp"), (-1, "#2ca02c", "conf")):
        s = e[e["opposing"] == opp]
        if len(s) == 0:
            continue
        x = ts.reindex(s["extremum_bar"]).to_numpy()                       # P1
        sym = np.where(s["polarity"] == 1, "triangle-down", "triangle-up")
        fig.add_trace(go.Scattergl(
            x=x, y=s["value"], mode="markers", name=f"{stream} {nm}",
            marker=dict(color=colour, size=size, symbol=sym,
                        line=dict(width=0.5, color="black")),
            legendgroup=stream, showlegend=True), row=row, col=1)


def plot_day_hawkes(d, ev, bar_from=None, bar_to=None, height=1400, width=2400):
    if bar_from is not None:
        d = d[d["bar_index"] >= bar_from]                                   # P3
        ev = ev[ev["extremum_bar"] >= bar_from]
    if bar_to is not None:
        d = d[d["bar_index"] <= bar_to]
        ev = ev[ev["extremum_bar"] <= bar_to]

    x = d["timestamp"]
    ts = pd.Series(d["timestamp"].to_numpy(), index=d["bar_index"].to_numpy())

    fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.025,
                        row_heights=[0.42, 0.18, 0.18, 0.22],
                        specs=[[{"secondary_y": False}]] * 4,
                        subplot_titles=["OHLC + JMA (dot colour = calibrated hazard)",
                                        "MNQ JMA derivatives",
                                        "TICK JMA derivatives",
                                        "Break probability"])

    fig.add_trace(go.Ohlc(x=x, open=d["Open"], high=d["High"], low=d["Low"],
                          close=d["Last"], name="OHLC", opacity=0.45,
                          increasing_line_color="#7f7f7f",
                          decreasing_line_color="#7f7f7f"), row=1, col=1)

    up = d["jma_leg_dir"].to_numpy() == 1
    dn = d["jma_leg_dir"].to_numpy() == -1
    jma = d["JMA"].to_numpy()
    fig.add_trace(go.Scattergl(x=x, y=_seg(jma, up), mode="lines", name="JMA up",
                               line=dict(width=2, color="#2ca02c")), row=1, col=1)
    fig.add_trace(go.Scattergl(x=x, y=_seg(jma, dn), mode="lines", name="JMA dn",
                               line=dict(width=2, color="#d62728")), row=1, col=1)

    if "hazard" in d.columns:
        fig.add_trace(go.Scattergl(
            x=x, y=jma, mode="markers", name="hazard",
            marker=dict(size=5, color=d["hazard"], colorscale="Turbo",
                        cmin=0.0, cmax=0.30, showscale=True,
                        colorbar=dict(title="p(next bar)", len=0.35, y=0.82))),
            row=1, col=1)

    _markers(fig, ev, "MNQ_JMA_SELF", ts, 1, size=9)

    for c, w in (("jmaD1", 1.6), ("jmaD2", 1.2)):
        fig.add_trace(go.Scattergl(x=x, y=d[c], mode="lines", name=c,
                                   line=dict(width=w)), row=2, col=1)
    _markers(fig, ev, "MNQ_D1", ts, 2)
    _markers(fig, ev, "MNQ_D2", ts, 2, size=5)

    for c, w in (("tickJmaD1", 1.6), ("tickJmaD2", 1.2)):
        fig.add_trace(go.Scattergl(x=x, y=d[c], mode="lines", name=c,
                                   line=dict(width=w)), row=3, col=1)
    _markers(fig, ev, "TICK_D1", ts, 3)
    _markers(fig, ev, "TICK_D2", ts, 3, size=5)

    if "hazard" in d.columns:
        fig.add_trace(go.Scattergl(x=x, y=d["hazard"], mode="lines",
                                   name="hazard (1 bar, cal)",
                                   line=dict(width=1.5, color="#1f77b4")), row=4, col=1)
    if "pbreak_frozen" in d.columns:
        fig.add_trace(go.Scattergl(x=x, y=d["pbreak_frozen"], mode="lines",
                                   name="pbreak frozen (raw)",                # P2
                                   line=dict(width=1.2, color="#ff7f0e", dash="dot")),
                      row=4, col=1)
    if "p8" in d.columns:
        fig.add_trace(go.Scattergl(x=x, y=d["p8"], mode="lines", name="pbreak<=8 (direct)",
                                   line=dict(width=1.2, color="#9467bd")), row=4, col=1)

    fig.add_hline(y=BASE_RATE, line=dict(color="black", dash="dash", width=1), row=4, col=1)

    tg = d[d["is_target"]]
    if len(tg):
        fig.add_trace(go.Scattergl(x=ts.reindex(tg["bar_index"] - 1).to_numpy(),
                                   y=np.zeros(len(tg)), mode="markers", name="JMA pivot",
                                   marker=dict(symbol="line-ns-open", size=10,
                                               color="black", line=dict(width=1))),
                      row=4, col=1)

    fig.update_layout(height=height, width=width, template="plotly_white",
                      hovermode="x unified", xaxis_rangeslider_visible=False,
                      legend=dict(groupclick="toggleitem"))
    fig.update_yaxes(title_text="Price", row=1, col=1)
    fig.update_yaxes(title_text="D1 / D2", row=2, col=1)
    fig.update_yaxes(title_text="TICK D1 / D2", row=3, col=1)
    fig.update_yaxes(title_text="P(break)", range=[0, 1], row=4, col=1)
    fig.update_xaxes(title_text="Time", row=4, col=1)
    fig.show()
    return fig

In [3]:
def add_hazard_bands(fig, d, thresh, row=1):
    hot = (d["hazard"] >= thresh).to_numpy()
    if not hot.any():
        return fig
    edges = np.diff(np.concatenate(([0], hot.view(np.int8), [0])))
    for s, e in zip(np.nonzero(edges == 1)[0], np.nonzero(edges == -1)[0] - 1):
        fig.add_vrect(x0=d["timestamp"].iloc[s], x1=d["timestamp"].iloc[e],
                      fillcolor="red", opacity=0.10, line_width=0, row=row, col=1)
    return fig

In [6]:
# Configuration: Style preferences
#plt.style.use('ggplot') # Good default for readability
pd.set_option("display.width", 400)      # total characters per line
pd.set_option("display.max_columns", 30) # prevent wrapping by limiting columns
pd.set_option("display.max_rows", 1000)

In [25]:
FRAME=6
HORIZON = 8
OUT_DIR='data/stage-2'

day = pd.Timestamp('2025-03-20').date()

rawFile= f'data/mnq-tick-oscillator-6sec.pqt'
raw = pd.read_parquet(rawFile)
raw = raw[raw["timestamp"].dt.date == pd.Timestamp(day).date()]
raw = raw[raw["timestamp"].dt.time >= pd.Timestamp("09:30").time()]

bars = pd.read_parquet(f'data/stage-0/bars_{FRAME}s.parquet')
bars = bars[bars["timestamp"].dt.date == pd.Timestamp(day).date()]
bars = bars[bars["timestamp"].dt.time >= pd.Timestamp("09:30").time()]

events = pd.read_parquet(f'data/stage-0/events_{FRAME}s.parquet')
events = events[events["timestamp"].dt.date == pd.Timestamp(day).date()]
events = events[events["timestamp"].dt.time >= pd.Timestamp("09:30").time()]

pred = pd.read_parquet(f"{OUT_DIR}/meta-iso-calibrated2-{HORIZON}h.parquet")
pred = pred[pred["timestamp"].dt.date == pd.Timestamp(day).date()]
pred = pred[pred["timestamp"].dt.time >= pd.Timestamp("09:30").time()]


In [26]:
print('\n-----------------------------------------------')
print(len(raw))
print(raw.head())
print('\n-----------------------------------------------')
print(len(bars))
print(bars.head())
print('\n-----------------------------------------------')
print(len(events))
print(events.head())
print('\n-----------------------------------------------')
print(len(pred))
print(pred.head())


-----------------------------------------------
3900
                  timestamp          Open          High       Low        Last           JMA       VEL    adpVEL  bolTopAdpVEL  bolMidAdpVEL  bolBotAdpVEL        RSX    tickRSX      jmaD1     jmaD2  bolTopJmaD2  bolMidJmaD2  bolBotJmaD2   tickJmaD1   tickJmaD2  bolTopTickJmaD2  bolMidTickJmaD2  bolBotTickJmaD2     tickJMA       date
3176810 2025-03-20 09:30:00  19785.343750  19787.750000  19770.00  19780.9375  19784.099609 -0.382195 -0.455477      0.494734     -0.339085     -1.172904 -89.096771 -33.962200  -0.765625  1.208984     1.364854    -0.010254    -1.385362 -102.979248  -73.155975        16.969612        -1.392377       -19.754368 -575.801147 2025-03-20
3176811 2025-03-20 09:30:06  19783.140625  19783.250000  19775.25  19779.0000  19782.748047 -0.384536 -0.445870      0.486274     -0.344772     -1.175817 -81.643143 -51.110668  -1.921875 -0.642578     1.364437     0.005039    -1.354358 -149.083160 -138.263062        20.078993  

In [27]:
COL = {"MNQ_D1": "jmaD1", "MNQ_D2": "jmaD2",
       "TICK_D1": "tickJmaD1", "TICK_D2": "tickJmaD2", "MNQ_JMA_SELF": "JMA"}

def verify(day, raw, bars, events, pred=None):
    r = raw[raw.timestamp.dt.date == pd.Timestamp(day).date()]
    b = bars[bars.date.astype(str) == day]
    e = events[events.date.astype(str) == day]
    out = {"n_raw": len(r), "n_bars": len(b)}

    m = r.merge(b[["timestamp", "jma", "is_target", "jma_leg_dir"]], on="timestamp")
    out["join_raw_bars"] = len(m)
    out["jma_exact"] = bool((m.JMA == m.jma).all())

    ts = pd.Index(r.timestamp)
    for s, col in COL.items():
        g = e[e.stream == s]
        v = r.set_index("timestamp")[col].reindex(g.extremum_timestamp)
        out[f"value_exact_{s}"] = bool((v.to_numpy() == g.value.to_numpy()).all())

    self_ts = set(e.loc[e.stream == "MNQ_JMA_SELF", "timestamp"])
    out["self_eq_is_target"] = self_ts == set(b.loc[b.is_target, "timestamp"])
    g = e[e.stream == "MNQ_JMA_SELF"]
    out["extremum_is_prev_bar"] = bool(
        (ts.get_indexer(g.timestamp) - ts.get_indexer(g.extremum_timestamp) == 1).all())

    if pred is not None:
        p = b.merge(pred, on="timestamp", suffixes=("", "_p"))
        out["join_bars_pred"] = len(p)
        out["is_target_agree"] = bool((p.is_target == p.is_target_p).all())
    return out

verify("2025-03-20", raw, bars, events, pred)

{'n_raw': 3900,
 'n_bars': 3900,
 'join_raw_bars': 3900,
 'jma_exact': True,
 'value_exact_MNQ_D1': True,
 'value_exact_MNQ_D2': True,
 'value_exact_TICK_D1': True,
 'value_exact_TICK_D2': True,
 'value_exact_MNQ_JMA_SELF': True,
 'self_eq_is_target': True,
 'extremum_is_prev_bar': True,
 'join_bars_pred': 3890,
 'is_target_agree': True}

In [28]:
print(pred.info())

<class 'pandas.DataFrame'>
RangeIndex: 3890 entries, 206453 to 210342
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   bar_index  3890 non-null   int64         
 1   timestamp  3890 non-null   datetime64[ns]
 2   is_target  3890 non-null   bool          
 3   p          3890 non-null   float32       
 4   p_cal      3890 non-null   float32       
 5   p_cal2     3890 non-null   float32       
dtypes: bool(1), datetime64[ns](1), float32(3), int64(1)
memory usage: 110.3 KB
None


In [ ]:
fig = plot_day_hawkes(day, raw, bar_from=b0 + 600, bar_to=b0 + 1100)

thresh = np.quantile(pred.p_cal, 0.90)      # top-decile state, ~0.20
add_hazard_bands(fig, d, thresh); fig.show()